In [1]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import warnings
from collections import deque
warnings.filterwarnings("ignore")

# ==========================================================
# LOAD MODEL
# ==========================================================
model_huruf        = joblib.load("model_bisindo_alfabet.pkl")
label_encoder_huruf = joblib.load("label_encoder_alfabet.pkl")

model_angka = joblib.load("model_number_xgboost.pkl")
label_encoder_angka = joblib.load("label_encoder_number.pkl")
model_kata         = joblib.load("model_kata_mlp.pkl")
scaler_kata        = joblib.load("scaler_kata_mlp.pkl")
label_encoder_kata = joblib.load("label_encoder_kata.pkl")

# ==========================================================
# THRESHOLD & SMOOTHING
# Naikkan CONFIDENCE_THRESHOLD kalau prediksi masih goyang
# ==========================================================
CONFIDENCE_THRESHOLD = 0.5   # di bawah ini tampilkan "-"
SMOOTH_WINDOW        = 10    # ambil prediksi mayoritas dari N frame terakhir

pred_history = deque(maxlen=SMOOTH_WINDOW)

# ==========================================================
# MEDIAPIPE
# ==========================================================
mp_hands    = mp.solutions.hands
mp_holistic = mp.solutions.holistic

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.5
)

holistic = mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

FACE_POINTS = [1, 33, 61, 199, 263]
POSE_POINTS = [11, 12, 13, 14, 15]
CHIN        = 152
NOSE        = 1
FOREHEAD    = 10    # dahi → Berpikir
CHEEK_RIGHT = 234   # pipi kanan → Tolong
CHEEK_LEFT  = 454   # pipi kiri
MOUTH       = 13    # mulut → Berdoa

mode = "huruf"
cap  = cv2.VideoCapture(0)

# ==========================================================
# FUNGSI SMOOTHING
# ==========================================================
def get_smooth_pred(new_pred):
    pred_history.append(new_pred)
    # ambil prediksi yang paling sering muncul
    from collections import Counter
    count = Counter(pred_history)
    return count.most_common(1)[0][0]

# ==========================================================
# FUNGSI EKSTRAK 1 TANGAN — huruf (46 fitur)
# ==========================================================
def extract_one_hand(hand_landmarks):
    lm  = hand_landmarks.landmark
    pts = np.array([[p.x, p.y] for p in lm])

    base = pts[0]
    pts  = pts - base

    max_val = np.max(np.abs(pts))
    if max_val != 0:
        pts = pts / max_val

    data = pts.flatten().tolist()

    def dist(a, b):
        return ((lm[a].x - lm[b].x)**2 + (lm[a].y - lm[b].y)**2)**0.5

    data.append(dist(4, 8))
    data.append(dist(8, 12))
    data.append(dist(12, 16))
    data.append(dist(16, 20))

    return data

# ==========================================================
# FUNGSI EKSTRAK KATA — 123 fitur (sama persis dengan CSV baru)
# ==========================================================
def extract_kata_features(result):
    data = []

    def normalize_landmarks(landmarks):
        pts = np.array([[lm.x, lm.y] for lm in landmarks])
        base = pts[0]
        pts  = pts - base
        mv   = np.max(np.abs(pts))
        if mv != 0:
            pts = pts / mv
        return pts.flatten().tolist()

    def pairwise_distances(landmarks):
        pairs = [(4, 8), (8, 12), (12, 16), (16, 20)]
        return [((landmarks[a].x - landmarks[b].x)**2 +
                 (landmarks[a].y - landmarks[b].y)**2)**0.5
                for a, b in pairs]

    def dist_xy(a, b):
        return ((a.x - b.x)**2 + (a.y - b.y)**2)**0.5

    # LEFT HAND
    if result.left_hand_landmarks:
        left = result.left_hand_landmarks.landmark
        data += normalize_landmarks(left)
        data += pairwise_distances(left)
    else:
        data += [0.0] * 46

    # RIGHT HAND
    if result.right_hand_landmarks:
        right = result.right_hand_landmarks.landmark
        data += normalize_landmarks(right)
        data += pairwise_distances(right)
    else:
        data += [0.0] * 46

    # FACE
    if result.face_landmarks:
        face = result.face_landmarks.landmark
        base = np.array([face[1].x, face[1].y])
        for i in FACE_POINTS:
            lm = face[i]
            pt = np.array([lm.x, lm.y]) - base
            data += pt.tolist()
    else:
        data += [0.0] * 10

    # POSE
    if result.pose_landmarks:
        pose = result.pose_landmarks.landmark
        base = np.array([pose[11].x, pose[11].y])
        for i in POSE_POINTS:
            lm = pose[i]
            pt = np.array([lm.x, lm.y]) - base
            data += pt.tolist()
    else:
        data += [0.0] * 10

    # CROSS FEATURES ASLI (2 fitur)
    extra = []
    if result.right_hand_landmarks and result.face_landmarks:
        hand = result.right_hand_landmarks.landmark
        face = result.face_landmarks.landmark
        extra.append(dist_xy(hand[8], face[NOSE]))
        extra.append(dist_xy(hand[4], face[NOSE]))
    else:
        extra += [0.0, 0.0]

    # FITUR TAMBAHAN BARU (9 fitur)
    if result.right_hand_landmarks and result.face_landmarks and result.pose_landmarks:
        rh   = result.right_hand_landmarks.landmark
        face = result.face_landmarks.landmark
        pose = result.pose_landmarks.landmark
        extra.append(dist_xy(rh[8], face[CHIN]))
        extra.append(dist_xy(rh[8], face[NOSE]))
        extra.append(dist_xy(rh[8], pose[12]))
        extra.append(dist_xy(rh[4], face[CHIN]))
    else:
        extra += [0.0] * 4

    if result.left_hand_landmarks and result.face_landmarks and result.pose_landmarks:
        lh   = result.left_hand_landmarks.landmark
        face = result.face_landmarks.landmark
        pose = result.pose_landmarks.landmark
        extra.append(dist_xy(lh[8], face[CHIN]))
        extra.append(dist_xy(lh[8], pose[11]))
        extra.append(dist_xy(lh[4], face[CHIN]))
    else:
        extra += [0.0] * 3

    both_hands = 1.0 if (result.right_hand_landmarks and result.left_hand_landmarks) else 0.0
    extra.append(both_hands)

    if result.right_hand_landmarks and result.face_landmarks:
        rh   = result.right_hand_landmarks.landmark
        face = result.face_landmarks.landmark
        y_rel = rh[8].y - face[NOSE].y
        extra.append(y_rel)
    else:
        extra.append(0.0)

    # FITUR TAMBAHAN BARU 2 (6 fitur)
    # Bedakan Tolong vs Berpikir, Apa vs Berdoa

    # Tangan kanan ke dahi, pipi, mulut
    if result.right_hand_landmarks and result.face_landmarks:
        rh   = result.right_hand_landmarks.landmark
        face = result.face_landmarks.landmark
        extra.append(dist_xy(rh[8], face[FOREHEAD]))     # telunjuk → dahi
        extra.append(dist_xy(rh[8], face[CHEEK_RIGHT]))  # telunjuk → pipi kanan
        extra.append(dist_xy(rh[8], face[MOUTH]))        # telunjuk → mulut
    else:
        extra += [0.0] * 3

    # Tangan kiri ke dahi, pipi, mulut
    if result.left_hand_landmarks and result.face_landmarks:
        lh   = result.left_hand_landmarks.landmark
        face = result.face_landmarks.landmark
        extra.append(dist_xy(lh[8], face[FOREHEAD]))
        extra.append(dist_xy(lh[8], face[CHEEK_LEFT]))
        extra.append(dist_xy(lh[8], face[MOUTH]))
    else:
        extra += [0.0] * 3

    data += extra
    return data  # 129 fitur

# ==========================================================
# LOOP CAMERA
# ==========================================================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    frame = cv2.resize(frame, (640, 480))
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    raw_pred  = "-"
    pred      = "-"
    conf_text = ""

    # ======================================================
    # MODE HURUF
    # ======================================================
    if mode == "huruf":
        result     = hands.process(rgb)
        empty_hand = [0.0] * 46
        left_data  = empty_hand.copy()
        right_data = empty_hand.copy()

        if result.multi_hand_landmarks:
            for hand_landmarks, handedness in zip(
                result.multi_hand_landmarks,
                result.multi_handedness
            ):
                label = handedness.classification[0].label
                if label == "Left":
                    left_data  = extract_one_hand(hand_landmarks)
                elif label == "Right":
                    right_data = extract_one_hand(hand_landmarks)

        data = left_data + right_data

        if result.multi_hand_landmarks:
            proba    = model_huruf.predict_proba(np.array(data).reshape(1, -1))[0]
            max_conf = np.max(proba)
            conf_text = f"{max_conf*100:.0f}%"
        
            if max_conf >= CONFIDENCE_THRESHOLD:
                pred_encoded = np.argmax(proba)
                raw_pred = label_encoder_huruf.inverse_transform([pred_encoded])[0]
                pred = get_smooth_pred(raw_pred)
            else:
                pred_history.clear()
                pred = "-"
        else:
            pred_history.clear()
            pred = "-"
    # ======================================================
    # MODE ANGKA
    # ======================================================
    elif mode == "angka":
        result = hands.process(rgb)
        hand   = None

        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]

        if hand:
            lm_list = hand.landmark
            data    = []
            base_x  = lm_list[0].x
            base_y  = lm_list[0].y

            for lm in lm_list:
                data.append(lm.x - base_x)
                data.append(lm.y - base_y)

            data = np.array(data)
            data = data - np.min(data)
            if np.max(data) != 0:
                data = data / np.max(data)
            data = data.tolist()

            def dist(a, b):
                return ((a.x - b.x)**2 + (a.y - b.y)**2)**0.5

            data.append(dist(lm_list[4],  lm_list[8]))
            data.append(dist(lm_list[8],  lm_list[12]))
            data.append(dist(lm_list[12], lm_list[16]))
            data.append(dist(lm_list[16], lm_list[20]))

            proba    = model_angka.predict_proba(np.array(data).reshape(1, -1))[0]
            max_conf = np.max(proba)
            conf_text = f"{max_conf*100:.0f}%"

            if max_conf >= CONFIDENCE_THRESHOLD:
                pred_encoded = np.argmax(proba)
                raw_pred = label_encoder_angka.inverse_transform([pred_encoded])[0]
                pred = get_smooth_pred(raw_pred)
            else:
                pred_history.clear()
                pred = "-"
        else:
            pred_history.clear()
            pred = "-"

    # ======================================================
    # MODE KATA — MLP + scaler + label encoder
    # ======================================================
    elif mode == "kata":
        result = holistic.process(rgb)
        data   = extract_kata_features(result)

        if len(data) == 129:
            hand_detected = (result.left_hand_landmarks or result.right_hand_landmarks)

            if hand_detected:
                # WAJIB: scale dulu sebelum masuk MLP
                data_scaled = scaler_kata.transform(np.array(data).reshape(1, -1))
                proba        = model_kata.predict_proba(data_scaled)[0]
                max_conf     = np.max(proba)
                conf_text    = f"{max_conf*100:.0f}%"

                if max_conf >= CONFIDENCE_THRESHOLD:
                    pred_encoded = np.argmax(proba)
                    raw_pred     = label_encoder_kata.inverse_transform([pred_encoded])[0]
                    pred         = get_smooth_pred(raw_pred)
                else:
                    pred_history.clear()
                    pred = "-"
            else:
                pred_history.clear()
                pred = "-"
        else:
            pred = f"err:{len(data)}"

    # ======================================================
    # TAMPILAN
    # ======================================================
    cv2.putText(frame, f"MODE  : {mode.upper()}",
                (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"HASIL : {pred}",
                (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    if conf_text:
        cv2.putText(frame, f"CONF  : {conf_text}",
                    (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 200, 0), 2)
    cv2.putText(frame, "A=Huruf  N=Angka  K=Kata  Q=Keluar",
                (10, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("Final Project Sign Language", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('a'):
        mode = "huruf"
        pred_history.clear()
    elif key == ord('n'):
        mode = "angka"
        pred_history.clear()
    elif key == ord('k'):
        mode = "kata"
        pred_history.clear()
    elif key == ord('q'):
        break

cap.release()
hands.close()
holistic.close()
cv2.destroyAllWindows()

In [5]:
import os
import joblib

model_kata = joblib.load("model_kata.pkl")
print("Classes:", model_kata.classes_)

# Cek nama-nama kata unik dari folder images
folder = "Dataset_Kata/train/images"  # sesuaikan path
namafile = os.listdir(folder)

# Ambil kata pertama dari nama file (sebelum tanda '-')
kata_unik = sorted(set(f.split('-')[0] for f in namafile))
print("Kata yang ada:", kata_unik)

Classes: [0 1 2 3 4 5 6]
Kata yang ada: ['berdoa', 'berjalan', 'berpikir', 'makan', 'mandi', 'saya', 'tidur']


In [3]:
import cv2
import mediapipe as mp
import os
import csv
import numpy as np
import random

DATASET_PATH = "dataset_angka"
OUTPUT = "angka3.csv"
TARGET_PER_CLASS = 400 

# Inisialisasi modul deteksi tangan dari MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.6  # minimum keyakinan deteksi = 60%
)

# ==========================================================
# AUGMENTASI
# ==========================================================
def augment_image(img):
    h, w = img.shape[:2]
    aug = img.copy()

    # 🔹 brightness & contrast 
    alpha = random.uniform(0.9, 1.1)
    beta  = random.randint(-15, 15)
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)

    # 🔹 rotasi kecil
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # 🔹 zoom ringan
    scale = random.uniform(0.95, 1.0)
    cx, cy = w // 2, h // 2
    nw, nh = int(w * scale), int(h * scale)
    x1 = max(cx - nw // 2, 0)
    y1 = max(cy - nh // 2, 0)
    x2 = min(x1 + nw, w)
    y2 = min(y1 + nh, h)
    aug = cv2.resize(aug[y1:y2, x1:x2], (w, h))

    return aug

# ==========================================================
# EKSTRAK LANDMARK 
# ==========================================================
def extract_landmarks(rgb_img):
    result = hands.process(rgb_img)

    if not result.multi_hand_landmarks:
        return None

    landmarks = result.multi_hand_landmarks[0]

    # pakai RELATIVE POSITION 
    base_x = landmarks.landmark[0].x
    base_y = landmarks.landmark[0].y

    data = []

    for lm in landmarks.landmark:
        data.append(lm.x - base_x)
        data.append(lm.y - base_y)

    # NORMALISASI
    data = np.array(data)
    data = data - np.min(data)
    if np.max(data) != 0:
        data = data / np.max(data)

    # FILTER DATA JELEK
    if np.max(data) - np.min(data) < 0.05:
        return None

    # TAMBAHAN FITUR JARAK JARI
    lm = landmarks.landmark
    def dist(a, b):
        return ((a.x - b.x)**2 + (a.y - b.y)**2) ** 0.5

    data = data.tolist()
    data.append(dist(lm[4], lm[8]))
    data.append(dist(lm[8], lm[12]))
    data.append(dist(lm[12], lm[16]))
    data.append(dist(lm[16], lm[20]))

    return data

# ==========================================================
# MAIN LOOP UTAMA PENGUMPULAN DATA
# ==========================================================
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)

    for label in sorted(os.listdir(DATASET_PATH)):
        folder = os.path.join(DATASET_PATH, label)
        if not os.path.isdir(folder):
            continue

        img_files = [
            os.path.join(folder, fn)
            for fn in os.listdir(folder)
            if fn.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
        ]

        if not img_files:
            print(f"[{label}] tidak ada gambar")
            continue

        print(f"[{label}] mulai dari {len(img_files)} gambar")

        # 🎯 balance ringan
        if str(label) == "6":
            TARGET = 200
        else:
            TARGET = TARGET_PER_CLASS

        berhasil = 0
        gagal = 0
        attempts = 0
        MAX_ATTEMPTS = TARGET * 10

        while berhasil < TARGET and attempts < MAX_ATTEMPTS:
            attempts += 1

            src_path = random.choice(img_files)
            img = cv2.imread(src_path)
            if img is None:
                continue

            img = cv2.resize(img, (640, 480))

            #augme ntasi lebih seimbang
            if random.random() < 0.5:
                processed = img.copy()
            else:
                processed = augment_image(img)

            rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
            data = extract_landmarks(rgb)

            if data:
                data.append(label)
                writer.writerow(data)
                berhasil += 1

                if berhasil % 50 == 0:
                    print(f"   {berhasil}/{TARGET}")
            else:
                gagal += 1

        print(f"[{label}] ✅ {berhasil}/{TARGET} | gagal: {gagal} | attempts: {attempts}\n")

hands.close()
print("✅ selesai:", OUTPUT)

[0] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[0] ✅ 400/400 | gagal: 327 | attempts: 727

[1] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[1] ✅ 400/400 | gagal: 25 | attempts: 425

[2] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[2] ✅ 400/400 | gagal: 44 | attempts: 444

[3] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[3] ✅ 400/400 | gagal: 0 | attempts: 400

[4] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[4] ✅ 400/400 | gagal: 2 | attempts: 402

[5] mulai dari 56 gambar
   50/400
   100/400
   150/400
   200/400
   250/400
   300/400
   350/400
   400/400
[5] ✅ 400/400 | gagal: 0 | attempts: 400

[6] mulai dari 56 gambar
   50/200
   100/200
   150/200
   200/200
[6] 

In [2]:
import joblib

model_huruf = joblib.load("model_alfabet.pkl")
model_angka = joblib.load("model_angka.pkl")
model_kata  = joblib.load("model_kata.pkl")

print(type(model_huruf))
print(type(model_angka))
print(type(model_kata))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>
